In [1]:
from pyiceberg.catalog import load_catalog
import duckdb

PROJECT = "/Users/Shared/academy/learner/projects/ethereum_iceberg"
catalog = load_catalog(
    "local",
    type="sql",
    uri=f"sqlite:///{PROJECT}/catalog/pyiceberg_catalog.db",
    warehouse=f"file://{PROJECT}/warehouse",
)
S3 = "s3://aws-public-blockchain/v1.0/eth"
con = duckdb.connect()
con.execute("set s3_region = 'us-east-2'")
catalog.list_namespaces()

[('raw',), ('ref',)]

In [2]:
import pathlib, datetime
blocks = catalog.load_table("raw.blocks")
def disk(table):
    folder = pathlib.Path(table.location().removeprefix("file://"))
    return {kind: len(list((folder / kind).rglob("*.*"))) for kind in ("data", "metadata")}
len(blocks.snapshots()), disk(blocks)

(10, {'data': 12, 'metadata': 37})

In [3]:
blocks.maintenance.expire_snapshots().older_than(datetime.datetime.now()).commit()
blocks = catalog.load_table("raw.blocks")
len(blocks.snapshots()), disk(blocks), [r["name"] for r in blocks.inspect.refs().to_pylist()]

(1, {'data': 12, 'metadata': 38}, ['main', 'after-cleanup'])

In [4]:
with blocks.transaction() as tx:
    tx.set_properties({
        "write.metadata.delete-after-commit.enabled": "true",
        "write.metadata.previous-versions-max": "5",
    })
blocks.properties

{'write.metadata.delete-after-commit.enabled': 'true',
 'write.metadata.previous-versions-max': '5'}